# quant-lens: GPT-2 small follow-up runs

Runtime > Change runtime type > **T4 GPU**, then run all cells. Follow-ups to the first run in `notebooks/run_gpt2.ipynb`:

1. **Robustness:** all four schemes again on a different slice of the Pile (`--doc-offset 5000`) with 4x the tokens.
2. **Fuller AWQ:** `awq_lite` with group-wise scales, and `awq_seq` (sequential calibration), per channel and group-wise, on the original slice.
3. **4-bit analyses:** layer ablation, fragility and steering at 4 bits, with three steering strengths.

The last cell zips everything except the per-feature tensors and downloads it.

In [ ]:
!git clone https://github.com/iamartyaa/quant-lens.git
%cd quant-lens
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Robustness: a different slice, 4x the tokens

In [ ]:
for scheme, extra in [('rtn', ''), ('rtn', '--group-size 128'), ('awq_lite', ''), ('w8a8', '--act-bits 8')]:
    name = scheme + ('_g128' if 'group' in extra else '')
    !time !python -m quant_lens.experiment --model gpt2 --scheme {scheme} {extra} --layers 2 6 10 --bits 8 6 4 3 \
        --doc-offset 5000 --n-docs 3000 --n-seqs 512 --out results/robust/gpt2_{name}

## 2. Fuller AWQ on the original slice

In [ ]:
!time !python -m quant_lens.experiment --model gpt2 --scheme awq_lite --group-size 128 --layers 2 6 10 --bits 8 6 4 3 --out results/gpt2_awq_lite_g128
!time !python -m quant_lens.experiment --model gpt2 --scheme awq_seq --layers 2 6 10 --bits 8 6 4 3 --out results/gpt2_awq_seq
!time !python -m quant_lens.experiment --model gpt2 --scheme awq_seq --group-size 128 --layers 2 6 10 --bits 8 6 4 3 --out results/gpt2_awq_seq_g128

## 3. Ablation, fragility and steering at 4 bits

In [ ]:
!time !python -m quant_lens.experiment --model gpt2 --scheme rtn --layers 0 1 2 3 4 5 6 7 8 9 10 11 --bits 8 6 4 3 --analysis-bits 4 \
    --ablate-layers --fragility --steering --steer-alphas 0.25 0.5 1.0 --out results/gpt2_rtn_4bit

## Reports and download

In [ ]:
import glob, subprocess
for d in sorted(glob.glob('results/gpt2_*') + glob.glob('results/robust/gpt2_*')):
    subprocess.run(['python', '-m', 'quant_lens.report', d], check=True)
!zip -qr results_followups.zip results -x '*/per_feature/*'
!ls -la results_followups.zip
from google.colab import files
files.download('results_followups.zip')